# Adapting Existing ViT Models to Use dHT

This notebook shows how to adapt pre-trained Vision Transformer models to use dHT tokenization.

## Approaches

There are several strategies to retrofit existing ViT models:

1. **Replace patch embedding**: Swap fixed patch embedding with dHT pipeline
2. **Fine-tune with frozen transformer**: Keep transformer weights, train only tokenizer
3. **Gradual unfreezing**: Start with frozen transformer, gradually unfreeze layers
4. **Knowledge distillation**: Train dHT model to match pre-trained ViT outputs

In [ ]:
import torch
import torch.nn as nn
from dht.nn.transformer import dHTEncoder
from dht.tok.tokenizer import dHTTokenizer
from dht.tok.extractor import dHTExtractor
from dht.tok.embedder import dHTEmbedder

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

## 1. Understanding ViT Architecture

Standard ViT consists of:
- Patch embedding (Conv2d or Linear)
- Positional embedding (learned or sinusoidal)
- Transformer blocks
- Classification head

We replace the patch + positional embedding with dHT components.

In [ ]:
# Example: Load a pre-trained ViT (using timm library)
# pip install timm
try:
    import timm
    
    # Load pre-trained ViT
    vit_model = timm.create_model('vit_base_patch16_224', pretrained=True)
    vit_model.eval()
    
    print('Pre-trained ViT loaded:')
    print(f'  Embed dim: {vit_model.embed_dim}')
    print(f'  Num heads: {vit_model.blocks[0].attn.num_heads}')
    print(f'  Depth: {len(vit_model.blocks)}')
except ImportError:
    print('timm not installed. Install with: pip install timm')
    print('Continuing with synthetic example...')
    vit_model = None

## 2. Strategy 1: Direct Replacement

Replace patch embedding while keeping transformer weights.

In [ ]:
class AdaptedViT(nn.Module):
    def __init__(self, original_vit, patch_size=16):
        super().__init__()
        
        # Extract parameters from original ViT
        self.embed_dim = original_vit.embed_dim if original_vit else 768
        self.patch_size = patch_size
        
        # Create dHT components
        self.tokenizer = dHTTokenizer(3, 8, compute_grad=True)
        self.extractor = dHTExtractor(patch_size)
        self.embedder = dHTEmbedder(self.embed_dim, patch_size, compute_grad=True)
        
        # Copy transformer blocks from original
        if original_vit:
            self.blocks = original_vit.blocks
            self.norm = original_vit.norm
            self.head = original_vit.head
        else:
            # Placeholder for demo
            self.blocks = nn.ModuleList([])
            self.norm = nn.LayerNorm(self.embed_dim)
            self.head = nn.Linear(self.embed_dim, 1000)
    
    def forward(self, x):
        # dHT tokenization pipeline
        tok_result = self.tokenizer(x)
        ext_result = self.extractor(tok_result)
        emb_result = self.embedder(ext_result)
        
        x, seg, amask = emb_result.fV, emb_result.seg, emb_result.amask
        
        # Transform through blocks
        for block in self.blocks:
            # Note: May need to adapt attention for variable sequences
            x = block(x)
        
        x = self.norm(x)
        return self.head(x[:, 0])  # CLS token

if vit_model:
    adapted = AdaptedViT(vit_model).to(device)
    print('\nAdapted ViT created with dHT tokenization')
else:
    print('\nSkipping due to missing timm library')

## 3. Strategy 2: Using dHTEncoder Directly

Build a new model using dHTEncoder and transfer weights.

In [ ]:
# Create dHT model matching ViT architecture
dht_model = dHTEncoder(
    embed_dim=768,
    patch_size=16,
    heads=12,
    depth=12,
    channels=3,
    compute_grad=True,
).to(device)

print('dHT Encoder created:')
print(f'  Compatible with ViT-Base architecture')
print(f'  Can load transformer weights from pre-trained ViT')

## 4. Weight Transfer

Transfer transformer block weights from pre-trained ViT:

In [ ]:
def transfer_transformer_weights(source_vit, target_dht):
    """
    Transfer transformer block weights from ViT to dHT model.
    
    Note: This assumes compatible architectures.
    May need adjustment for different ViT implementations.
    """
    if source_vit is None:
        print('No source model provided')
        return
    
    # Transfer block weights
    for i, (src_block, tgt_block) in enumerate(zip(source_vit.blocks, target_dht.blocks)):
        # Copy attention weights
        tgt_block.attn.load_state_dict(src_block.attn.state_dict(), strict=False)
        
        # Copy MLP weights
        tgt_block.mlp.load_state_dict(src_block.mlp.state_dict(), strict=False)
        
        # Copy norms
        tgt_block.norm1.load_state_dict(src_block.norm1.state_dict())
        tgt_block.norm2.load_state_dict(src_block.norm2.state_dict())
    
    # Transfer final norm
    target_dht.norm.load_state_dict(source_vit.norm.state_dict())
    
    print(f'Transferred weights for {len(source_vit.blocks)} transformer blocks')

# Example usage:
if vit_model:
    transfer_transformer_weights(vit_model, dht_model)
else:
    print('Skipping weight transfer (no pre-trained model)')

## 5. Fine-tuning Strategies

After weight transfer, choose a fine-tuning strategy:

In [ ]:
# Strategy A: Freeze transformer, train tokenizer only
def freeze_transformer(model):
    model.freeze_blocks()
    for p in model.norm.parameters():
        p.requires_grad = False

# Strategy B: Gradual unfreezing
def unfreeze_last_n_blocks(model, n=3):
    for block in model.blocks[-n:]:
        for p in block.parameters():
            p.requires_grad = True

# Strategy C: Train everything
def unfreeze_all(model):
    model.unfreeze_blocks()
    for p in model.parameters():
        p.requires_grad = True

# Example: Freeze transformer
freeze_transformer(dht_model)
trainable = sum(p.numel() for p in dht_model.parameters() if p.requires_grad)
total = sum(p.numel() for p in dht_model.parameters())
print(f'\nStrategy A - Frozen Transformer:')
print(f'  Trainable: {trainable/1e6:.1f}M / {total/1e6:.1f}M parameters')

# Unfreeze last 3 blocks
unfreeze_last_n_blocks(dht_model, n=3)
trainable = sum(p.numel() for p in dht_model.parameters() if p.requires_grad)
print(f'\nStrategy B - Gradual Unfreezing (last 3 blocks):')
print(f'  Trainable: {trainable/1e6:.1f}M / {total/1e6:.1f}M parameters')

## 6. Resolution Adaptation

Adapt to different input resolutions:

In [ ]:
# Test with different resolutions
resolutions = [224, 384, 512]

dht_model.eval()
for res in resolutions:
    test_img = torch.randn(1, 3, res, res).to(device)
    with torch.no_grad():
        output = dht_model(test_img)
    print(f'Resolution {res}x{res}: Output shape {output.fV.shape}')

## 7. Comparison: ViT vs dHT-ViT

Let's compare efficiency and token counts:

In [ ]:
import time

# Test image
test_img = torch.randn(4, 3, 224, 224).to(device)

# Standard ViT
if vit_model:
    vit_model.eval()
    with torch.no_grad():
        start = time.time()
        vit_out = vit_model(test_img)
        vit_time = time.time() - start
    print(f'Standard ViT:')
    print(f'  Tokens: 196 (fixed)')
    print(f'  Time: {vit_time*1000:.1f}ms')

# dHT-ViT
dht_model.eval()
with torch.no_grad():
    start = time.time()
    dht_out = dht_model(test_img)
    dht_time = time.time() - start

avg_tokens = dht_out.amask.sum(dim=1).float().mean().item()
print(f'\ndHT-ViT:')
print(f'  Tokens: {avg_tokens:.1f} (adaptive)')
print(f'  Time: {dht_time*1000:.1f}ms')
print(f'\nToken reduction: {(1 - avg_tokens/196)*100:.1f}%')

## Summary

This notebook covered:
1. Understanding ViT architecture
2. Direct replacement of patch embedding
3. Using dHTEncoder with weight transfer
4. Fine-tuning strategies
5. Resolution adaptation
6. Efficiency comparison

### Key Steps to Adapt Your ViT:

1. Create dHTEncoder with matching architecture
2. Transfer transformer weights
3. Choose fine-tuning strategy
4. Train on your dataset
5. Evaluate and compare

### Next: Video Models (Notebook 06)